# Data Preprocessing

## Import Dependencies

In [23]:
import os
import warnings

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

from mappers import to_numeric

warnings.filterwarnings("ignore")

## Data Loading

In [24]:
root = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
path = os.path.join(root, "raw.csv")

dataset = pd.read_csv(filepath_or_buffer=path)

## Data Cleaning

#### Remove unnecessary features
1. `Category URL`

2. `Service URL`

3. `Offer URL`

4. `Offer Name`

5. `Owner URL`

6. `Owner Name`


In [25]:
unnecessary_features = ["Category URL", "Service URL", "Offer URL", "Offer Name", "Owner URL", "Owner Name"]

dataset.drop(columns=unnecessary_features, inplace=True)

#### Convert text-based values to numeric values
1. Time: `Duration`, `Offer Response Time`, and `Owner Response Time`.

2. Percentage: `Owner Completion Rate`.

3. Boolean: `Owner Verified`.

4. Money: `Price`.


In [26]:
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].mask(dataset["Owner Completion Rate"] == "لم يحسب بعد")

dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].replace("[\%,]", "", regex=True).astype(float)

dataset["Owner Verified"] = dataset["Owner Verified"].astype(int)

dataset["Price"] = dataset["Price"].replace("[\$,]", "", regex=True).astype(float)

dataset = to_numeric(dataset, columns=["Duration", "Offer Response Time", "Owner Response Time"])

#### Remove the missing values

In [27]:
dataset = dataset[~dataset.isna().any(axis=1)]

dataset.shape

(5944, 20)

## Encoding
1. `Owner Level`

2. `Category Name`

3. `Service Name`

#### Ordinal Encoding for `Owner Level` 

In [28]:
top_prices = (dataset.groupby("Owner Level", group_keys=False).apply(lambda x: x.nlargest(1, "Price")))

top_prices["Frequency"] = top_prices.apply(lambda row: dataset[(dataset["Owner Level"] == row["Owner Level"]) & (dataset["Price"] == row["Price"])].shape[0], axis=1)

order = (top_prices[["Owner Level", "Price", "Frequency"]].sort_values(by=["Price", "Frequency"])["Owner Level"].unique())

ordinal = OrdinalEncoder(categories=[order])

dataset["Owner Level"] = ordinal.fit_transform(dataset[["Owner Level"]])

#### One-Hot Encoding for `Category Name` and `Service Name`

In [29]:
categorical_features = ["Category Name", "Service Name"]

one_hot = OneHotEncoder()

encoded = one_hot.fit_transform(dataset[categorical_features])

encoded = pd.DataFrame(encoded.toarray(), columns=[col.split("_", 1)[-1] for col in one_hot.get_feature_names_out(categorical_features)])

encoded.reset_index(drop=True, inplace=True)
dataset.reset_index(drop=True, inplace=True)

dataset = pd.concat([dataset, encoded], axis=1).drop(categorical_features, axis=1)

## Save the cleaned regression dataset

In [30]:
path = os.path.join(root, "regression.csv")

dataset.to_csv(path_or_buf=path, index=False)

## Save the cleaned classification dataset

In [31]:
def categorize_price(price):
    """
    TODO
    :param price: 
    :return: 
    """
    if 5 <= price <= 15:
        return "Bronze"
    elif 20 <= price <= 35:
        return "Silver"
    elif 40 <= price <= 50:
        return "Gold"
    
dataset["Price Category"] = dataset.pop("Price").apply(categorize_price)

In [32]:
path = os.path.join(root, "classification.csv")

dataset.to_csv(path_or_buf=path, index=False)